# B5 — Sector analysis

Tests 1–3 broken down by industry. **The three hypotheses below were stated
before any sector result was seen, and are the confirmatory tests.** Every
other sector comparison in this notebook is descriptive.

| | Hypothesis | Mechanism | Prediction |
|---|---|---|---|
| **H1** | Technology shows more speculative retail demand | Tech receives the most media attention | Tech has a higher retail share, OTM share and short-dated share, and lower retail returns; the tech effect **shrinks** once attention (abnormal pre-announcement volume) is controlled for |
| **H2** | Earnings are a minor event for pharma & biotech | Biotech's largest news is regulatory (FDA decisions, trial readouts), not earnings | A smaller announcement move relative to normal volatility, a milder near-the-money IV crush, and weaker information in pre-announcement flow |
| **H3** | Flow is more informative in energy & industrials | Earnings there are more forecastable from public data (commodity prices, input costs, backlogs) | Flow × energy_industrials interaction **< 0** for announcement returns and surprises, especially for professional flow |

**Design.** Sector differences are tested directly — sector dummies and
flow × sector interactions within one regression — never by comparing
separate per-sector regressions (a significant result in one group and an
insignificant one in another does not show that they differ). Every test
controls for firm size and year-quarter fixed effects, with standard errors
clustered by firm and announcement date. Tests pool 2016–2025; eras are a
secondary check.

**Reading the results.** Each hypothesis has several predefined outcomes.
Judge a hypothesis by whether its outcomes point the same way, not by any
single p-value.

**Requires A1–A3** (the Test 1 event panel and the saved contract panel).

## Setup

In [ ]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl

## Build the sector panel

One row per firm-event: the Test 1 panel plus the firm's sector (from CRSP
NAICS, as of the announcement date), the attention proxy, pre-event
volatility, the announcement move relative to normal volatility, and the
option measures from the contract panel.

In [ ]:
from analysis.informed_trading import build_event_panel
from analysis.option_returns import PANEL_PATH
from analysis.sector_analysis import build_sector_panel

panel = build_event_panel(verbose=False)
contracts = pl.read_parquet(PANEL_PATH)
sp = build_sector_panel(panel, contracts)

## Check the sector assignments

The most frequent firms in each sector. Before reading any result, confirm
familiar names land where they should — for example large chip and software
firms in Technology, the oil majors in Energy, the big drugmakers in Pharma &
biotech. Known borderline cases: e-commerce is Consumer & retail, carmakers
are Industrials, media and telecom are Other.

In [ ]:
from analysis.sector_analysis import sector_check

with pl.Config(tbl_rows=-1, fmt_str_lengths=120, tbl_width_chars=200):
    print(sector_check(sp))

## Descriptive profile (not a test)

Medians of each measure by sector, for context.

In [ ]:
from analysis.sector_analysis import sector_profile

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=220):
    print(sector_profile(sp))

## H1 — Technology and speculative demand

For each measure: the tech coefficient with a size control, then on the same
sample with the attention proxy added. **H1 predicts** a positive tech
coefficient on the demand measures, a negative one on retail returns, and a
tech coefficient that shrinks once attention is added. A tech effect that
survives the attention control is a sector effect; one that disappears is an
attention effect.

In [ ]:
from analysis.sector_analysis import test_h1_tech

with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200):
    print(test_h1_tech(sp).select("test", "outcome", "term", "n", "coef", "se", "p"))

## H2 — Biotech: earnings as a minor event

The biotech coefficient on the size of the announcement event, with a size
control. **H2 predicts** negative coefficients on `event_ratio` and
`abs_car`, and **positive** coefficients on `atm_iv_change` (a milder crush
is a less negative change). The flow interactions test the third prediction:
a biotech interaction of the opposite sign to the flow effect.

Caution: the dispersion control scales by |MEANEST| with a 0.05 floor, which
behaves oddly for loss-making firms, so biotech's flow results carry that
caveat.

In [ ]:
from analysis.sector_analysis import test_h2_biotech, test_flow_interactions

inter = test_flow_interactions(sp)
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200):
    print(test_h2_biotech(sp).select("outcome", "term", "n", "coef", "se", "p"))
    print(inter.filter(pl.col("hypothesis") == "H2").select("test", "outcome", "term", "n", "coef", "se", "p"))

## H3 — Energy & industrials: informed flow

`dv ~ flow_z + flow_z × energy_industrials + energy_industrials + controls`,
separately for retail and professional flow. **H3 predicts** a negative
interaction for announcement returns (`car01`) and surprises
(`surprise_scaled`), strongest for professional flow. The `flow_z` row is the
flow effect in all other sectors.

In [ ]:
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200):
    print(inter.filter(pl.col("hypothesis") == "H3").select("test", "outcome", "term", "n", "coef", "se", "p"))

## Secondary: by era

The same tests within each era. Samples are much smaller here, especially
for energy and biotech, so treat these as a consistency check on the pooled
results rather than as tests in their own right.

In [ ]:
from analysis.informed_trading import ERAS
from analysis.sector_analysis import run_sector_tests

by_era = pl.concat([run_sector_tests(sp.filter(pl.col("era") == e)).with_columns(pl.lit(e).alias("era"))
                    for e in ERAS], how="diagonal_relaxed")
key_terms = ["tech", "biotech", "flow_z:energy_industrials", "flow_z:biotech"]
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=220):
    print(by_era.filter(pl.col("term").is_in(key_terms) & (pl.col("test") != "tech, + attention"))
          .select("era", "hypothesis", "test", "outcome", "term", "n", "coef", "p"))

## Write results

`table12_sector_profile.csv` (descriptive), `table12b_sector_membership.csv`
(the sector check) and `table13_sector_hypotheses.csv` (every H1–H3 test) to
`results/`.

In [ ]:
from analysis.sector_analysis import write_sector_results

sector_results = write_sector_results(sp)